In [1]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
import re
import emoji
from nltk.corpus import stopwords
import contractions
import ftfy
from nltk import pos_tag
from nltk.corpus import wordnet
import pandas as pd
import importlib.resources as pkg_resources
from symspellpy import SymSpell, Verbosity

In [2]:
def ensure_nltk_data():
    resources = {
        'punkt': 'tokenizers/punkt',
        'wordnet': 'corpora/wordnet',
        'omw-1.4': 'corpora/omw-1.4',
        'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng',
        'stopwords': 'corpora/stopwords'
    }
    for r, path in resources.items():
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(r)

ensure_nltk_data()
stop_words = set(stopwords.words('english'))
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
with pkg_resources.path("symspellpy", "frequency_dictionary_en_82_765.txt") as dictionary_path:
    sym_spell.load_dictionary(str(dictionary_path), term_index=0, count_index=1)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yasmi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\yasmi\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
important_words = {
    "not", "no", "never",
    "but", "however",
    "very", "really",
    "more", "less"
}

clean text with emoji:
<pre>  remove unwanted characters except ! and ?
<pre>  sequences of ! and ? (2 or more in a row) = exclamation_strong
<pre>  single exclamation = exclamation
<pre>  remove single question marks
<pre>  remove sequences of other repeated punctuation
<pre>  Keep only words, spaces, hyphens, and emoji/exclamation placeholders

In [ ]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# --- Tag extraction ---
def extract_car_model(text):
    # if not isinstance(text, str):
    #     return f"{text}"
    # match = re.search(r'\b(Tesla|Nissan|Skoda|Rivian|Chevy|Ford|Lucid|Volkswagen|toyota|sentra|BMW|ID Buzz|hyundai|mustang|kia|Volvo|audi|lexus|jeep|Porsche|Volkswagens|Mazda|mercedes|range rover|renault  )\b', text, re.IGNORECASE)
    # return match.group(0) if match else 
    return "Other"

def extract_topic(text):
    # if not isinstance(text, str):
    #     return f"{text}"
    # topics = ['battery', 'range', 'price', 'charging', 'autopilot', 'design', 'performance', 'reliability', 'fuel']
    # for t in topics:
    #     if re.search(rf'\b{t}\b', text, re.IGNORECASE):
    #         return t
    return "Other"

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def clean_text_with_emoji(text):
    if not isinstance(text, str):
        return str(text)
    
    text = ftfy.fix_text(text)
    text = contractions.fix(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"[:_@#$%^&*()\"\'<>/\\|~`+=;,.]", " ", text)
    text = emoji.demojize(text)
    text = re.sub(r":([a-zA-Z0-9_+-]+):", r" \1_emoji ", text)
    text = re.sub(r"([!?]{2,})", " exclamation_strong ", text)
    text = re.sub(r"(?<!\!)\!(?!\!)", " exclamation ", text)
    text = re.sub(r"\?", " ", text)
    text = re.sub(r"([*\-=_]{2,})", " ", text)
    text = re.sub(r"[^\w\s\-]", "", text)
    tokens = word_tokenize(text)
    tagged_tokens = pos_tag(tokens)
    clean_tokens = [lemmatizer.lemmatize(t.lower(), get_wordnet_pos(p)) for t, p in tagged_tokens]
    filtered_tokens = [
        t for t in clean_tokens
        if t not in stop_words or t in important_words
    ]
    corrected_tokens = []
    for token in filtered_tokens:
        if token.endswith("_emoji") or token.startswith("exclamation"):
            # skip placeholders
            corrected_tokens.append(token)
        else:
            suggestions = sym_spell.lookup(token, Verbosity.CLOSEST, max_edit_distance=2)
            corrected_tokens.append(suggestions[0].term if suggestions else token)
    
    return " ".join(corrected_tokens)

In [5]:
df = pd.read_csv(r"..\data\fetched_yt_comments.csv")
print(df.shape)
df.head()

(8307, 7)


,video_id,video_title,author,comment,likes,published_at,category
0,PghQPGacrlI,Best SUVs tested: BMW v Porsche v Mercedes v A...,@carwow,Sell your car for free with Carwow: https://bi...,133,2024-07-29T10:12:23Z,suv
1,PghQPGacrlI,Best SUVs tested: BMW v Porsche v Mercedes v A...,@silvanelsire5971,wheres xc90,0,2026-03-17T08:23:53Z,suv
2,PghQPGacrlI,Best SUVs tested: BMW v Porsche v Mercedes v A...,@kingofthebridge8339,Land cruiser is better than all of them.,0,2026-03-12T20:49:12Z,suv
3,PghQPGacrlI,Best SUVs tested: BMW v Porsche v Mercedes v A...,@andrescardonagutierrez6732,Where’s the Volvo xC90,0,2026-03-04T21:58:20Z,suv
4,PghQPGacrlI,Best SUVs tested: BMW v Porsche v Mercedes v A...,@johns8702,I can see u only praise bmw .. not good,0,2026-02-25T04:19:54Z,suv


In [6]:
df['clean_comment'] = df['comment'].apply(clean_text_with_emoji)
df['car_model'] = df['comment'].apply(extract_car_model)
df['topic'] = df['comment'].apply(extract_topic)

In [7]:
df.sample(100)

,video_id,video_title,author,comment,likes,published_at,category,clean_comment,car_model,topic
3279,OVS0BMMrU5M,Kia K4 vs. Toyota Corolla Hybrid vs. Honda Civ...,@LOUIS-hb6zt,K4 Hybrid is coming up.,3,2025-02-07T09:50:02Z,sedan,of hybrid come,Other,Other
2967,UQGTUonY5qs,Best Compact Car: 8-Car Mega Comparison,@dirkdiggler164,The one I wanted was a Mazda CX30 genuinely lo...,2,2025-06-01T05:20:38Z,sedan,one want mazda cx30 genuinely lovely look righ...,Mazda,Other
3920,avd1GdpB_2A,The Differences Between Muscle And Pony Cars,@devonriley110,BS right off the bat. The 64.5 was NOT the fir...,0,2017-07-10T20:06:30Z,coupe,a right bat of a not first pony car barracuda ...,Mustang,Other
4194,avd1GdpB_2A,The Differences Between Muscle And Pony Cars,@lg763,ARE YOU HURT,1,2017-03-26T00:27:05Z,coupe,hurt,Other,Other
2962,UQGTUonY5qs,Best Compact Car: 8-Car Mega Comparison,@cliftondelaney1389,Those cars from the late 70s had 400 - 500hp &...,0,2025-06-01T05:52:35Z,sedan,car late of 400 a 500hp still achieve mpg,Other,Other
...,...,...,...,...,...,...,...,...,...,...
7362,J-CcwVxcfaQ,Audi A5 vs Mercedes C-Class Coupe vs BMW 4 Ser...,@Bertofornell93,"none of those, the giulia veloce without a dou...",0,2020-11-27T15:45:22Z,coupe,none julia veloce without doubt best exclamati...,Other,Other
1064,PghQPGacrlI,Best SUVs tested: BMW v Porsche v Mercedes v A...,@adriannotfound6497,What the fuck did I just watch...never compare...,0,2024-07-29T17:03:54Z,suv,fuck watch never compare sun future,Other,Other
2812,UQGTUonY5qs,Best Compact Car: 8-Car Mega Comparison,@stevestann595,Tired of automotive journalists reviewing car...,0,2025-07-16T08:32:47Z,sedan,tired automotive journalist review car talk sp...,Other,price
5912,nMn_ueG80Zw,"$50,000 Used Question // GT350 vs Alfa 4C vs B...",@austinvukmer3754,For that amount of money I’ll always stick wit...,0,2025-06-19T02:35:40Z,coupe,amount money always stick of is corvette close...,Other,Other


In [8]:
df.to_csv(r"..\data\full_clean_yt_comments.csv",header=True)